# Sprint 5 – Notebook 5: Data Validation


## 1. What is Data Validation?

### Definition
Data validation checks whether information follows expected rules before it is used. It identifies incorrect, incomplete, or inconsistent entries in a dataset. Validation can happen when data is collected or during preprocessing. It helps prevent unreliable results and supports better decisions.

### Business Example
An online store receives customer orders from several branches. Some orders may contain missing customer IDs, negative quantities, or incorrect prices. The store checks each record before accepting it for billing. Invalid orders are flagged for correction instead of silently being processed.

### AI/ML Use
Machine learning models learn patterns from the data supplied to them. Incorrect records can introduce noise and misleading relationships. Validation helps maintain trustworthy training and prediction datasets. It also catches problems early when new data arrives after deployment.

### Python Example

In [1]:
import pandas as pd

df = pd.DataFrame({"Age": [24, -3, 35], "Salary": [30000, 45000, -100]})
valid = (df["Age"] >= 0) & (df["Salary"] >= 0)
print("Valid rows:", df[valid])
print("Invalid rows:", df[~valid])

Valid rows:    Age  Salary
0   24   30000
Invalid rows:    Age  Salary
1   -3   45000
2   35    -100


### Explanation
The dataset contains both correct and incorrect records. The two conditions require non-negative age and salary values. The `&` operator combines both checks into one validation rule. The code prints valid and invalid rows separately so problems can be reviewed.

## 2. Range Validation

### Definition
Range validation checks whether numerical values fall within allowed limits. These limits come from real-world meaning or business requirements. A percentage usually needs to be between 0 and 100. Values outside the allowed range are flagged as invalid.

### Business Example
A school stores exam percentages for its students. A score of 85 is valid, while 125 is impossible on a 100-point scale. The school checks the allowed range before publishing results. This prevents incorrect scores from affecting rankings and reports.

### AI/ML Use
Features with impossible values can distort model training and predictions. Range checks are useful for ages, prices, ratings, and percentages. They reduce the chance of extreme entry errors being treated as meaningful data. Valid ranges should be chosen using domain knowledge.

### Python Example

In [3]:
import pandas as pd

df = pd.DataFrame({"Age": [25, -5, 42], "Percentage": [78, 110, 65]})
valid = df["Age"].ge(0) & df["Percentage"].between(0, 100)
print(df.assign(Valid=valid))

   Age  Percentage  Valid
0   25          78   True
1   -5         110  False
2   42          65   True


### Explanation
`ge(0)` checks that age is not negative. `between(0, 100)` checks the percentage range, including both endpoints. A row passes only when both conditions are true. The new `Valid` column clearly shows which records require correction.

## 3. Type Validation

### Definition
Type validation checks whether a value has the expected data type. Numerical fields should contain numbers rather than unrelated text. Dates should be stored in a form that can be interpreted as dates. Correct types allow calculations, comparisons, and filtering to work properly.

### Business Example
A payroll system receives salary values from an employee form. One record contains `45000`, while another contains `forty thousand`. The second value cannot be used directly in salary calculations. The system flags it for correction before preparing payroll.

### AI/ML Use
ML preprocessing often requires numeric columns for scaling and model fitting. Text stored in numeric fields can cause conversion errors. Type validation identifies such problems before a pipeline runs. This makes training and prediction more reliable.

### Python Example

In [5]:
import pandas as pd

df = pd.DataFrame({"Salary": ["30000", "unknown", "45000"]})
converted = pd.to_numeric(df["Salary"], errors="coerce")
print(df.assign(Valid_Type=converted.notna()))

    Salary  Valid_Type
0    30000        True
1  unknown       False
2    45000        True


### Explanation
`to_numeric()` tries to convert salary strings into numbers. The option `errors="coerce"` turns invalid text into missing values. `notna()` marks successfully converted entries as valid. The original data remains visible for investigation.

## 4. Format Validation

### Definition
Format validation checks whether text follows a required pattern or structure. Common examples include email addresses, phone numbers, and employee IDs. A value may be a string but still have an incorrect format. Pattern checks catch these issues before the data is used.

### Business Example
A company requires employee codes such as `EMP-102`. An entry like `102EMP` does not follow the expected structure. The HR team validates the codes before creating employee accounts. This reduces mismatches when records are shared across systems.

### AI/ML Use
Identifiers and contact fields often support joining datasets and linking predictions to users. Incorrect formats can break these connections. Format validation makes input records more consistent. The rule should match the actual requirements of the application.

### Python Example

In [7]:
import pandas as pd

df = pd.DataFrame({"Employee_ID": ["EMP-101", "101-EMP", "EMP-205"]})
valid = df["Employee_ID"].str.fullmatch(r"EMP-\d{3}")
print(df.assign(Valid_Format=valid))

  Employee_ID  Valid_Format
0     EMP-101          True
1     101-EMP         False
2     EMP-205          True


### Explanation
`str.fullmatch()` checks whether the entire value follows a pattern. The pattern requires `EMP-` followed by exactly three digits. A code with a different order fails validation. This example checks structure, not whether the employee actually exists.

## 5. Category Validation

### Definition
Category validation checks whether a value belongs to an approved list. It is useful for fields such as gender, department, and product category. Unexpected spellings or new labels may be mistakes or legitimate changes. Invalid categories should be reviewed rather than automatically deleted.

### Business Example
A registration form accepts the categories `Male`, `Female`, and `Other` for a particular field. A record containing `Mle` does not match the accepted options. The system flags the spelling for correction. If the organization expands its accepted categories, the rule must also be updated.

### AI/ML Use
Categorical encoders expect consistent category labels. Variations such as `Sales` and `sales` may be treated as separate groups. Category checks prevent accidental duplication of categories. They also help detect unseen labels in prediction data.

### Python Example

In [9]:
import pandas as pd

df = pd.DataFrame({"Gender": ["Male", "Female", "Mle", "Other"]})
accepted = {"Male", "Female", "Other"}
print(df.assign(Valid_Category=df["Gender"].isin(accepted)))

   Gender  Valid_Category
0    Male            True
1  Female            True
2     Mle           False
3   Other            True


### Explanation
`isin()` compares each value with the accepted category set. The incorrect spelling `Mle` is marked as invalid. Valid categories are not changed by the check. The accepted list should reflect the organization’s actual policy and data requirements.

## 6. Null Validation

### Definition
Null validation checks whether required fields contain missing values. Some columns must always be present, while others can be optional. A blank or missing identifier may make a record unusable. Validation rules should distinguish required fields from optional ones.

### Business Example
A delivery company requires an order ID and a delivery address. An order without an address cannot be shipped correctly. The company flags the incomplete order for follow-up. An optional delivery note can remain empty without causing an error.

### AI/ML Use
Missing target labels can prevent supervised learning on affected rows. Missing features may require imputation or another suitable strategy. Null validation identifies where missing data occurs. It does not mean every missing value must be filled with the mean.

### Python Example

In [11]:
import pandas as pd

df = pd.DataFrame({"Order_ID": [101, 102, 103],
                   "Address": ["Hassan", None, "Bengaluru"]})
valid = df[["Order_ID", "Address"]].notna().all(axis=1)
print(df.assign(Valid_Required_Fields=valid))

   Order_ID    Address  Valid_Required_Fields
0       101     Hassan                   True
1       102       None                  False
2       103  Bengaluru                   True


### Explanation
`notna()` checks whether each required field has a value. `all(axis=1)` requires both fields to be present in each row. The order with a missing address is marked invalid. The code flags the issue without guessing a replacement address.

## 7. Business Rule Validation

### Definition
Business rule validation checks whether data follows rules specific to an organization. These rules often combine multiple columns rather than checking one field. For example, salary cannot be negative and a discount may have a maximum limit. Rules should be documented so everyone interprets them consistently.

### Business Example
A company records employee salaries and bonuses. It does not allow negative salary amounts. It also requires bonuses to be non-negative. Payroll entries that break either rule are sent for review before payment.

### AI/ML Use
Business rules prevent logically impossible training examples. They can also protect live ML systems from accepting invalid inputs. Domain experts help define which combinations are acceptable. A useful rule must be based on business reality, not an arbitrary assumption.

### Python Example

In [13]:
import pandas as pd

df = pd.DataFrame({"Salary": [30000, -1000, 45000],
                   "Bonus": [2000, 500, -300]})
valid = (df["Salary"] >= 0) & (df["Bonus"] >= 0)
print(df.assign(Valid_Business_Rule=valid))

   Salary  Bonus  Valid_Business_Rule
0   30000   2000                 True
1   -1000    500                False
2   45000   -300                False


### Explanation
The rule requires salary and bonus to be non-negative. Parentheses keep each comparison easy to read. The combined condition marks any row that breaks a rule as invalid. The results can be reviewed before payroll calculations or ML training.

## 8. Referential Validation

### Definition
Referential validation checks whether an identifier refers to a valid record in another dataset. It is often used with customer IDs, product IDs, and department IDs. A transaction should not point to a nonexistent customer. These checks maintain relationships between connected tables.

### Business Example
A store keeps a customer table and an orders table. Every order must belong to a customer registered in the customer table. An order with an unknown customer ID is flagged. Staff can then investigate whether the customer record is missing or the ID was entered incorrectly.

### AI/ML Use
ML projects often merge data from several sources. Invalid references can produce missing joined values or incorrect feature relationships. Referential checks help confirm that linked records belong together. They are especially important before building features from multiple tables.

### Python Example

In [15]:
import pandas as pd

customers = pd.DataFrame({"Customer_ID": [101, 102, 103]})
orders = pd.DataFrame({"Order_ID": [1, 2, 3],
                       "Customer_ID": [101, 999, 103]})
valid = orders["Customer_ID"].isin(customers["Customer_ID"])
print(orders.assign(Valid_Customer=valid))

   Order_ID  Customer_ID  Valid_Customer
0         1          101            True
1         2          999           False
2         3          103            True


### Explanation
The customer table provides the list of valid customer IDs. `isin()` checks whether each order references one of those IDs. Customer ID `999` is flagged because it is absent from the customer table. The check does not delete the order automatically.

## 9. Date Validation

### Definition
Date validation checks whether date values are correctly formatted and logically possible. It can detect invalid calendar dates and incorrect chronological order. For example, a joining date should not be earlier than a date of birth. Dates should be converted carefully before comparisons.

### Business Example
An HR database stores employee birth dates and joining dates. A record showing that an employee joined before being born is impossible. The company flags this record for correction. Validating dates also helps calculate reliable employee tenure.

### AI/ML Use
Date fields are used to build time-based features such as age and tenure. Incorrect dates can create negative durations or misleading patterns. Date validation improves the quality of these derived features. It is also important when dividing time-series data into training and testing periods.

### Python Example

In [17]:
import pandas as pd

df = pd.DataFrame({"Birth_Date": ["2000-05-10", "2002-08-15"],
                   "Joining_Date": ["2024-06-01", "2001-01-01"]})
df["Birth_Date"] = pd.to_datetime(df["Birth_Date"], errors="coerce")
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"], errors="coerce")
df["Valid_Date"] = (df["Joining_Date"] >= df["Birth_Date"]) & df[["Birth_Date", "Joining_Date"]].notna().all(axis=1)
print(df)

  Birth_Date Joining_Date  Valid_Date
0 2000-05-10   2024-06-01        True
1 2002-08-15   2001-01-01       False


### Explanation
`to_datetime()` converts the text columns into comparable date values. Invalid date strings would become missing values because of `errors="coerce"`. The comparison checks that joining occurs on or after birth. The validity column identifies the record with impossible date order.

## 10. Constraint Validation

### Definition
Constraint validation checks whether data satisfies conditions that must always hold. Constraints may include uniqueness, non-null values, and allowed numeric limits. Unlike a single field check, a constraint can apply across an entire table. These rules protect the consistency of stored information.

### Business Example
A bank requires every account number to be unique. It also requires each account balance to be non-negative for a particular account product. Two rows with the same account number violate the uniqueness rule. A negative balance violates the balance rule and must be reviewed.

### AI/ML Use
Duplicate identifiers can cause accidental double counting during model training. Invalid values can also introduce errors into engineered features. Constraint validation helps keep records consistent across batches. Checks should be aligned with the rules of the actual data system.

### Python Example

In [19]:
import pandas as pd

df = pd.DataFrame({"Account_ID": [101, 102, 101],
                   "Balance": [5000, 2000, -300]})
valid = ~df["Account_ID"].duplicated(keep=False) & df["Balance"].ge(0)
print(df.assign(Valid_Constraints=valid))

   Account_ID  Balance  Valid_Constraints
0         101     5000              False
1         102     2000               True
2         101     -300              False


### Explanation
`duplicated(keep=False)` identifies every row with a repeated account ID. The `~` symbol reverses that result so only unique IDs pass. `ge(0)` checks that the balance is not negative. Both rules must pass for a record to satisfy the constraints.